# Code 2.6.3: Backpropagating through the 2-2-1 worked example

The worked MLP example of Section 2.3 written with `Scalar` operations. The sigmoid is built from `exp`, addition, and division, and the loss from `log`.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 2.6.1
import math

class Scalar:
    __slots__ = ("value", "grad", "inputs", "op")

    def __init__(self, value, inputs=(), op=""):
        self.value = float(value)
        self.grad = 0.0
        self.inputs = inputs      # tuple of (parent Scalar, d self / d parent)
        self.op = op              # operation name, for printing and debugging

    def __repr__(self):
        return f"Scalar({self.value:.4f}, grad={self.grad:.4f}, op='{self.op}')"

    # ---- building blocks -------------------------------------------------
    @staticmethod
    def lift(x):
        return x if isinstance(x, Scalar) else Scalar(x)

    def __add__(self, other):
        other = Scalar.lift(other)
        return Scalar(self.value + other.value, ((self, 1.0), (other, 1.0)), "+")

    def __mul__(self, other):
        other = Scalar.lift(other)
        return Scalar(self.value * other.value,
                      ((self, other.value), (other, self.value)), "*")

    def __pow__(self, k):
        assert isinstance(k, (int, float)), "only constant exponents"
        return Scalar(self.value ** k, ((self, k * self.value ** (k - 1)),), f"^{k}")

    def exp(self):
        e = math.exp(self.value)
        return Scalar(e, ((self, e),), "exp")

    def log(self):
        return Scalar(math.log(self.value), ((self, 1.0 / self.value),), "log")

    def tanh(self):
        t = math.tanh(self.value)
        return Scalar(t, ((self, 1.0 - t * t),), "tanh")

    def relu(self):
        return Scalar(max(0.0, self.value), ((self, float(self.value > 0)),), "relu")

    # ---- derived operations (built from the ones above) ------------------
    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-Scalar.lift(other))

    def __truediv__(self, other):
        return self * Scalar.lift(other) ** -1

    __radd__ = __add__
    __rmul__ = __mul__

    def __rsub__(self, other):
        return Scalar.lift(other) - self

    def __rtruediv__(self, other):
        return Scalar.lift(other) / self

    # ---- reverse pass -------------------------------------------------------
    def backward(self):
        """Fill in .grad for every Scalar that this one depends on."""
        order, visited = [], set()
        stack = [(self, False)]
        while stack:                       # iterative depth-first topological sort
            node, children_done = stack.pop()
            if children_done:
                order.append(node)
                continue
            if id(node) in visited:
                continue
            visited.add(id(node))
            stack.append((node, True))
            for parent, _ in node.inputs:
                if id(parent) not in visited:
                    stack.append((parent, False))
        self.grad = 1.0
        for node in reversed(order):       # outputs before inputs
            for parent, local in node.inputs:
                parent.grad += local * node.grad


In [ ]:
x = [1.0, 0.5]
W1 = [[Scalar(0.5), Scalar(-0.3)],
      [Scalar(0.8), Scalar(0.2)]]
b1 = [Scalar(0.0), Scalar(0.1)]
W2 = [Scalar(1.0), Scalar(-1.5)]
b2 = Scalar(0.2)

h = [(x[0] * W1[0][j] + x[1] * W1[1][j] + b1[j]).tanh() for j in range(2)]
z2 = h[0] * W2[0] + h[1] * W2[1] + b2
p = 1 / (1 + (-z2).exp())                       # sigmoid built from exp, add, divide
loss = -(p.log())                               # binary cross-entropy with y = 1
loss.backward()

print(f"loss = {loss.value:.4f}")
print("dW1 =", [[round(W1[i][j].grad, 4) for j in range(2)] for i in range(2)])
print("db1 =", [round(v.grad, 4) for v in b1])
print("dW2 =", [round(v.grad, 4) for v in W2], " db2 =", round(b2.grad, 4))


**Expected output**

Output:

```text
loss = 0.2960
dW1 = [[-0.1247, 0.3805], [-0.0624, 0.1902]]
db1 = [-0.1247, 0.3805]
dW2 = [-0.1835, 0.0255]  db2 = -0.2562
```
